In [1]:
pip install "mesa==2.4.0"

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 12.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ------------------------------------ --- 7.3/8.1 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 31.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --------------------------- ------------ 8.4/12.3 MB 40.0 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 35.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 38.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------------------------------ --- 8.9/9.7 MB 46


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#!pip uninstall mesa -y
#!pip install "mesa==2.4.0"

Found existing installation: Mesa 3.5.1
Uninstalling Mesa-3.5.1:
  Successfully uninstalled Mesa-3.5.1
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached charset_normalizer-3.4.6-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Usi

# 기본

In [2]:
import mesa # 전체를 불러와서 경로 문제를 원천 봉쇄합니다.

# 1. UGV 에이전트 클래스
class UGVAgent(mesa.Agent): # 1. 설계도(틀) 이름 정하기
    def __init__(self, model): # 2. 붕어빵 찍어낼 때 초기 설정
        super().__init__(model) # 3. 기본 규격 맞추기
        self.battery = 100 # 4. 개별 속성 부여

    def step(self):
        # 1. 내 현재 위치 확인 (쟁반 위 어디에 있나?)
        current_pos = self.pos
        if current_pos is None: return 
        
        # 2. 날씨 정보 가져오기 (기상청 지도 확인)
        weather = self.model.weather_map.get(current_pos, "Clear")
        
        # 3. 현재 턴수 확인 (시계 확인)
        step_count = getattr(self.model, "steps", 0)
        
        # 4. 상태 보고 (로그 출력)
        print(f"[턴 {step_count}] UGV 위치: {current_pos} | 날씨: {weather} | 배터리: {self.battery}")

        # 5. 다음 갈 곳 계산 (오른쪽 위 대각선)
        x, y = current_pos
        new_x = min(x + 1, self.model.grid.width - 1)
        new_y = min(y + 1, self.model.grid.height - 1)
        
        # 6. 실제 이동 실행
        self.model.grid.move_agent(self, (new_x, new_y))
        
        # 7. 배터리 소모 계산
        if weather == "Severe":
            self.battery -= 20
        else:
            self.battery -= 10

# 2. 모델 클래스 (세상)
class UGVModel(mesa.Model):
    def __init__(self, width, height):
        super().__init__()
        # 1. 공간(쟁반) 만들기
        self.grid = mesa.space.MultiGrid(width, height, torus=False)
        
        # 2. 날씨 데이터 (기상청 역할)
        self.weather_map = {(2, 2): "Severe", (3, 3): "Severe"}

        # 3. 에이전트 생성 및 맵에 올리기
        ugv = UGVAgent(self)
        self.grid.place_agent(ugv, (0, 0))

    def step(self):
        # 1. 맵 위의 모든 요원들에게 명령 하달
        for agent in self.agents:
            agent.step()
        
        # 2. 세상의 시계 1턴 증가
#        if hasattr(self, "steps"):
#            self.steps += 1

# 3. 실행 테스트
sim_model = UGVModel(width=5, height=5)
print("--- 통합 호환 버전 시뮬레이션 시작 ---")
for i in range(5):
    sim_model.step()

KeyboardInterrupt: 

# 장애물(20), 악천후(20)

In [102]:
import random
import mesa

# 1. UGV 에이전트 클래스
class UGVAgent(mesa.Agent): # 1. 설계도(틀) 이름 정하기
    def __init__(self, model): # 2. 붕어빵 찍어낼 때 초기 설정
        super().__init__(model) # 3. 기본 규격 맞추기
        self.battery = 100 # 4. 개별 속성 부여

    def step(self):
        # 1. 내 현재 위치 확인 (쟁반 위 어디에 있나?)
        current_pos = self.pos
        if current_pos is None: return 
        
        # 2. 날씨 정보 가져오기 (기상청 지도 확인)
        weather = self.model.weather_map.get(current_pos, "Clear")
        
        # 3. 현재 턴수 확인 (시계 확인)
        step_count = getattr(self.model, "steps", 0)
        
        # 4. 상태 보고 (로그 출력)
        print(f"[턴 {step_count}] UGV 위치: {current_pos} | 날씨: {weather} | 배터리: {self.battery}")

        # 5. 다음 갈 곳 계산 (오른쪽 위 대각선)
        x, y = current_pos
        new_x = min(x + 1, self.model.grid.width - 1)
        new_y = min(y + 1, self.model.grid.height - 1)
        
        # 6. 실제 이동 실행
        self.model.grid.move_agent(self, (new_x, new_y))
        
        # 7. 배터리 소모 계산
        if weather == "Severe":
            self.battery -= 20
        else:
            self.battery -= 10

class UGVModel(mesa.Model):
    def __init__(self, width, height):
        super().__init__()
        self.grid = mesa.space.MultiGrid(width, height, torus=False)
        self.steps = 0
        
        # 1. 빈 공간 리스트 만들기 (장애물과 날씨를 배치할 후보지)
        all_coords = [(x, y) for x in range(width) for y in range(height)]
        all_coords.remove((0, 0))  # UGV가 시작하는 (0,0)은 비워둡니다.
        
        # 2. 장애물(Obstacle) 배치하기 (20칸 무작위 선택)
        self.obstacles = random.sample(all_coords, 20)
        for coord in self.obstacles:
            all_coords.remove(coord) # 장애물이 들어간 곳은 날씨 배치 후보에서 제외
            
        # 3. 악천후(Severe) 배치하기 (남은 공간 중 20칸 무작위 선택)
        severe_weather_coords = random.sample(all_coords, 20)
        self.weather_map = {coord: "Severe" for coord in severe_weather_coords}
        
        # 4. UGV 배치
        ugv = UGVAgent(self)
        self.grid.place_agent(ugv, (0, 0))

    def step(self):
        # 1. 맵 위의 모든 에이전트(UGV 등)를 불러와서 각각의 step()을 실행시킵니다.
        for agent in self.agents:
            agent.step()
        
        # 2. 세상의 시간(턴)을 1 올립니다.
        self.steps += 1

    def show_map(self):
        print("\n=== 현재 시뮬레이션 맵 상황 (X: 장애물, =: 악천후, U: UGV) ===")
        for y in range(self.grid.height - 1, -1, -1): # 위에서 아래로 출력
            row = ""
            for x in range(self.grid.width):
                coord = (x, y)
                
                # 1. UGV가 있는가?
                if len(self.grid.get_cell_list_contents([coord])) > 0:
                    row += "[U]"
                # 2. 장애물인가?
                elif coord in self.obstacles:
                    row += "[X]"
                # 3. 악천후인가?
                elif self.weather_map.get(coord) == "Severe":
                    row += "[=]"
                # 4. 빈칸인가?
                else:
                    row += "[ ]"
            print(row)
        print("=========================================================\n")

# 실행부
model = UGVModel(10, 10)
model.show_map() # 세상을 만들자마자 지도를 보여줘!

# 5턴 진행 후 다시 확인
for _ in range(5):
    model.step()
    
model.show_map() # 5턴 후에 UGV가 어디까지 갔는지 다시 보여줘!


=== 현재 시뮬레이션 맵 상황 (X: 장애물, =: 악천후, U: UGV) ===
[ ][ ][X][X][=][ ][=][X][=][ ]
[X][ ][ ][ ][ ][ ][ ][=][X][ ]
[X][ ][ ][ ][ ][ ][ ][=][X][ ]
[ ][ ][X][=][=][ ][ ][ ][ ][ ]
[X][ ][=][ ][ ][X][ ][ ][=][ ]
[X][ ][ ][X][ ][X][ ][X][X][ ]
[ ][=][ ][ ][ ][=][ ][ ][ ][ ]
[ ][ ][=][ ][ ][ ][ ][=][ ][=]
[=][=][=][X][ ][ ][=][=][X][X]
[U][ ][ ][ ][ ][=][X][ ][ ][X]

[턴 1] UGV 위치: (0, 0) | 날씨: Clear | 배터리: 100
[턴 3] UGV 위치: (1, 1) | 날씨: Severe | 배터리: 90
[턴 5] UGV 위치: (2, 2) | 날씨: Severe | 배터리: 70
[턴 7] UGV 위치: (3, 3) | 날씨: Clear | 배터리: 50
[턴 9] UGV 위치: (4, 4) | 날씨: Clear | 배터리: 40

=== 현재 시뮬레이션 맵 상황 (X: 장애물, =: 악천후, U: UGV) ===
[ ][ ][X][X][=][ ][=][X][=][ ]
[X][ ][ ][ ][ ][ ][ ][=][X][ ]
[X][ ][ ][ ][ ][ ][ ][=][X][ ]
[ ][ ][X][=][=][ ][ ][ ][ ][ ]
[X][ ][=][ ][ ][U][ ][ ][=][ ]
[X][ ][ ][X][ ][X][ ][X][X][ ]
[ ][=][ ][ ][ ][=][ ][ ][ ][ ]
[ ][ ][=][ ][ ][ ][ ][=][ ][=]
[=][=][=][X][ ][ ][=][=][X][X]
[ ][ ][ ][ ][ ][=][X][ ][ ][X]



In [7]:
import mesa
import random
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mesa.datacollection import DataCollector
import pandas as pd

# 1. 에이전트 설계
# 1. 에이전트 설계
class UGVAgent(mesa.Agent):
    def __init__(self, unique_id, model):      # unique_id 추가!
        super().__init__(unique_id, model)   # unique_id 추가!
        self.battery = 200 
        self.destination = (9, 9)
        # (이하 코드 동일)
        
        try:
            # ⭐️ [수정 1] weight='weight' 옵션 추가: 거리가 멀어도 비용이 싼 길을 찾습니다!
            self.planned_path = nx.shortest_path(
                self.model.graph, 
                source=(0, 0), 
                target=self.destination, 
                weight='weight'
            )
            self.full_path = list(self.planned_path) 
            self.planned_path.pop(0) 
        except nx.NetworkXNoPath:
            self.planned_path = []
            self.full_path = [] 

    # ⭐️ 해결: 이제 def __init__ 과 def step 이 완전히 같은 세로줄(들여쓰기)에 있습니다!
    def step(self):
        if not self.planned_path: return
        
        current_pos = self.pos
        next_pos = self.planned_path.pop(0)
        self.model.grid.move_agent(self, next_pos)
        
        # ⭐️ 1. 대각선 이동 여부에 따른 기본 배터리 소모량 계산
        dx = abs(next_pos[0] - current_pos[0])
        dy = abs(next_pos[1] - current_pos[1])
        is_diagonal = (dx == 1 and dy == 1)
        
        # 직선은 10, 대각선은 14 소모
        base_battery_consume = 14 if is_diagonal else 10 
        
        # ⭐️ 2. 날씨(진흙탕)에 따른 2배 페널티 적용
        weather = self.model.weather_map.get(next_pos, "Clear")
        actual_consume = base_battery_consume * 2 if weather == "Severe" else base_battery_consume
        
        self.battery -= actual_consume
        print(f"[Turn {self.model.steps}] {current_pos} -> {next_pos} | 소모량: -{actual_consume} | 잔여 배터리: {self.battery}")

# 2. 모델(세상) 설계
class UGVModel(mesa.Model):
    def __init__(self, width, height):
        super().__init__()
        self.grid = mesa.space.MultiGrid(width, height, torus=False)
        self.steps = 0
        
        all_coords = [(x, y) for x in range(width) for y in range(height)]
        all_coords.remove((0, 0))
        all_coords.remove((width-1, height-1))
        
        self.obstacles = random.sample(all_coords, 20)
        for coord in self.obstacles: all_coords.remove(coord)
        
        # ⭐️ [수정 2] 날씨 세팅을 도로망 구축보다 '먼저' 하도록 위치를 위로 올렸습니다.
        self.weather_map = {coord: "Severe" for coord in random.sample(all_coords, 20)}
            
        # 경로 계산을 위한 그래프 생성
        self.graph = nx.Graph()
        for x in range(width):
            for y in range(height):
                if (x, y) not in self.obstacles:
                    self.graph.add_node((x, y))
        
        for node in self.graph.nodes:
            x, y = node
            for dx, dy in [(0,1),(0,-1),(1,0),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)]:
                neighbor = (x+dx, y+dy)
                if neighbor in self.graph.nodes:
                    
                    # 1. 실제 배터리 소모량과 똑같이 기본 비용 설정 (직선 10, 대각선 14)
                    is_diagonal = (dx != 0 and dy != 0)
                    base_cost = 14 if is_diagonal else 10 
                    
                    # 2. 악천후 시 배터리가 2배 닳으므로, 가중치도 정확히 2배만 적용!
                    if self.weather_map.get(node) == "Severe" or self.weather_map.get(neighbor) == "Severe":
                        cost = base_cost * 2
                    else:
                        cost = base_cost
                    
                    self.graph.add_edge(node, neighbor, weight=cost)
        
        # ⭐️ (수정 후)
        ugv = UGVAgent(1, self)  # 숫자 1 (고유 ID) 추가!
        self.grid.place_agent(ugv, (0, 0))

        # ==========================================
        # ⭐️ 추가: 데이터 수집기(DataCollector) 설정 ⭐️
        self.datacollector = DataCollector(
            agent_reporters={
                "X좌표": lambda a: a.pos[0] if a.pos else None,
                "Y좌표": lambda a: a.pos[1] if a.pos else None,
                "잔여배터리": "battery", # agent.battery 값을 바로 가져옵니다.
                "현재날씨": lambda a: a.model.weather_map.get(a.pos, "Clear") if a.pos else "Clear"
            }
        )
        # ==========================================

    def step(self):
        # 1. 에이전트들 이동
        for agent in self.agents:
            agent.step()
            
        # 2. 턴 증가
        self.steps += 1
        
        # 3. ⭐️ 추가: 모든 이동이 끝난 후 현재 상태 기록 ⭐️
        self.datacollector.collect(self)

    def plot_map(self):
        grid_matrix = np.zeros((self.grid.height, self.grid.width))
        for coord, w in self.weather_map.items(): grid_matrix[coord[1], coord[0]] = 1 # 비
        for coord in self.obstacles: grid_matrix[coord[1], coord[0]] = 2 # 장애물
        grid_matrix[9, 9] = 3 # 목적지
        
        for agent in self.agents:
            if agent.pos: grid_matrix[agent.pos[1], agent.pos[0]] = 4 # UGV

        cmap = ListedColormap(['white', 'skyblue', 'black', 'green', 'red'])
        
        plt.figure(figsize=(5, 5))
        plt.imshow(grid_matrix, cmap=cmap, origin='lower', vmin=0, vmax=4)

        for agent in self.agents:
            if hasattr(agent, 'full_path') and agent.full_path:
                path_x = [pos[0] for pos in agent.full_path]
                path_y = [pos[1] for pos in agent.full_path]
                plt.plot(path_x, path_y, color='yellow', linestyle='--', linewidth=2, marker='o', markersize=4)
                
        plt.title(f"Simulation Turn: {self.steps}")
        plt.show() 

# --- 실행부 ---
model = UGVModel(10, 10)
model.plot_map() # 초기 맵 확인

# 목적지에 도착할 때까지 최대 15턴 진행
for _ in range(15):
    model.step()

model.plot_map() # 최종 맵 확인

# ==========================================
# ⭐️ 추가: 수집된 데이터를 Pandas 데이터프레임으로 변환 ⭐️
df = model.datacollector.get_agent_vars_dataframe()

# 인덱스를 깔끔하게 정리 (Mesa는 기본적으로 Step과 AgentID를 다중 인덱스로 씁니다)
df = df.reset_index().rename(columns={"Step": "턴", "AgentID": "에이전트ID"})

print("\n📊 [시뮬레이션 주행 기록 데이터]")
print(df.head(15)) # 15턴까지의 기록 출력
# ==========================================

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [8]:
! pip uninstall pandas -y
! pip install pandas
! pip uninstall mesa

Found existing installation: pandas 3.0.1
Uninstalling pandas-3.0.1:
  Successfully uninstalled pandas-3.0.1
  Using cached pandas-3.0.1-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.1-cp312-cp312-win_amd64.whl (9.7 MB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C


In [ ]:
pip install --upgrade numpy matplotlib
pip install "mesa==2.4.0"

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   -- ------------------------------------- 2.6/36.5 MB 12.6 MB/s eta 0:00:03
   ----- ---------------------------------- 5.2/36.5 MB 12.8 MB/s eta 0:00:03
   ------- -------------------------------- 6.6/36.5 MB 10.6 MB/s eta 0:00:03
   --------- ------------------------------ 8.4/36.5 MB 10.2 MB/s eta 0:00:03
   ---------- ----------------------------- 10.0/36.5 MB 10.5 MB/s eta 0:00:03
   ------------- -------------------------- 12.1/36.5 MB 9.8 MB/s eta 0:00:03
   --------------- ------------------------ 14.4/36.5 MB 10.1 MB/s eta 0:00:03
   ------------------ --------------------- 16.8/36.5 MB 10.1 MB/s eta 0:00:02
   ------------------- -------------------- 18.1/36.5 MB 9.7 MB/s eta 0:00:02
   ---------------------- ----------------- 20.2/36.5 MB 9.7 MB/s eta 0:00:02
   ------------------------ --------------- 22.3/36.5 MB 9.7 MB/s eta 0:00:02
   --------------------------- ------------ 24.9/36.5 MB 9.9 MB/s eta


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
